<a href="https://colab.research.google.com/github/ereznaamansapienza/mnlp/blob/banana/mnlp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [1]:
import torch
import gc
import os
import json
import numpy as np
import pandas as pd
import random
from datasets import load_dataset, Dataset, load_from_disk

from transformers import AutoModelForCausalLM, AutoTokenizer

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from typing import Dict, List, Tuple, Callable, Optional
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path


# Setup

In [2]:
torch.cuda.empty_cache()
gc.collect()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

ds_url = "sapienzanlp-course-materials/hw-mnlp-2026"

cpu


In [3]:
SEED = 42

def set_seed(seed: int = 42):
  random.seed(seed)
  np.random.seed(seed)

  torch.manual_seed(seed)

  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

set_seed(SEED)

# Data Module

In [4]:
class DataModule:
    def __init__(self, dev_size: float = 0.2, seed: int = 42):
        self.ds       = None
        self.dev_size = dev_size
        self.seed     = seed

    # ------------------------------------------------------------------ #
    #  Loading                                                             #
    # ------------------------------------------------------------------ #

    def load(self, url: str):
        self.ds = load_dataset(url)
        return self.ds

    def to_dataframe(self, split_name: str) -> pd.DataFrame:
        return pd.DataFrame(self.ds[split_name])

    # ------------------------------------------------------------------ #
    #  Splits                                                              #
    # ------------------------------------------------------------------ #

    def build_train_val_split(self) -> Tuple[pd.DataFrame, pd.DataFrame]:
        train_df_full = self.to_dataframe("train")
        train_df, val_df = train_test_split(
            train_df_full,
            test_size=self.dev_size,
            random_state=self.seed,
            shuffle=True,
        )
        return train_df, val_df

    def build_cv_splits(
        self,
        n_folds: int = 5,
        split_name: str = "train",
        stratify_col: str = "wikipedia_title",
    ) -> List[Tuple[pd.DataFrame, pd.DataFrame]]:
        df = self.to_dataframe(split_name).reset_index(drop=True)

        if stratify_col is not None:
            kf         = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=self.seed)
            split_iter = kf.split(df, df[stratify_col])
        else:
            kf         = KFold(n_splits=n_folds, shuffle=True, random_state=self.seed)
            split_iter = kf.split(df)

        folds = []
        for train_idx, val_idx in split_iter:
            folds.append((
                df.iloc[train_idx].reset_index(drop=True),
                df.iloc[val_idx].reset_index(drop=True),
            ))
        return folds

    # ------------------------------------------------------------------ #
    #  Dataset builders — convenience wrappers                            #
    # ------------------------------------------------------------------ #

    def build_full_train_dataset(
        self,
        dataset_factory: Callable,
        **factory_kwargs,
    ) -> Tuple[pd.DataFrame, Dataset]:
        """Builds a dataset from the entire training split with no val holdout."""
        train_df = self.to_dataframe("train")
        dataset  = dataset_factory(train_df, **factory_kwargs)
        return train_df, dataset

    def build_train_val_datasets(
        self,
        dataset_factory: Callable,
        **factory_kwargs,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, Dataset, Dataset]:
        """Splits train into train/val and builds datasets for both."""
        train_df, val_df = self.build_train_val_split()
        train_dataset    = dataset_factory(train_df, **factory_kwargs)
        val_dataset      = dataset_factory(val_df,   **factory_kwargs)
        return train_df, val_df, train_dataset, val_dataset


# Question Answering LLM

In [5]:
class QALLM():
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.model = self.load_model()
        self.tokenizer = self.load_tokenizer()

    def load_model(self):
        return AutoModelForCausalLM.from_pretrained(
            self.model_name,
            torch_dtype="auto",
            device_map="auto"
        )

    def load_tokenizer(self):
        return AutoTokenizer.from_pretrained(self.model_name)

    def prepare_model_input(self, prompt, thinking):
        messages = [
            {"role": "user", "content": prompt}
        ]
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=thinking # Switches between thinking and non-thinking modes. Default is True.
        )
        return self.tokenizer([text], return_tensors="pt").to(self.model.device)

    def generate_answer(self, prompt, thinking):
        model_inputs = self.prepare_model_input(prompt, thinking)

        # conduct text completion
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=32768
        )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

        # parsing thinking content
        try:
            # rindex finding 151668 (</think>)
            index = len(output_ids) - output_ids[::-1].index(151668)
        except ValueError:
            index = 0

        thinking_content = self.tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
        content = self.tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

        return content, thinking_content


# Evaluation

## Evaluation Metrics

## Evaluating LLM

# Experiments

In [6]:
data_module = DataModule(dev_size=0.2)
ds = data_module.load("sapienzanlp-course-materials/hw-mnlp-2026")

train_df, val_df = data_module.build_train_val_split()
test_df = data_module.to_dataframe("test")
blind_df = data_module.to_dataframe("blind")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [10]:
train_df.head()

,wikipedia_title,wikidata_id,query,query_id,candidate_chunks,n_candidates,answer,answer_pos,short_answer
1467,Kolhapur State,Q657946,when was kolhapur state merged into bombay pro...,pabaPqShLcfF,[Kolhapur State or Kolhapur Maratha Kingdom (1...,7,The last ruler of Kolhapur was HH Maharaja Chh...,4,[1 March 1949]
5768,Good Deeds,Q5582519,where was tyler perry's good deeds filmed,z350gAmIWOEj,[Good Deeds is a 2012 American romantic drama ...,15,Good Deeds was produced by Perry's 34th Street...,8,[Principal photography took place in Atlanta]
5714,Gas metal arc welding,Q1765723,what is the function of wire feeder in mig mag...,ZH8tBXiWbFyh,"[Gas metal arc welding (GMAW), sometimes refer...",40,The wire feed unit supplies the electrode to t...,9,"[supplies the electrode to the work, driving i..."
1578,Long Island Sound,Q867460,how deep is the water in long island sound,MiRBsyVCKmxb,[Long Island Sound is a tidal estuary of the A...,61,Long Island Sound is a tidal estuary of the At...,0,[from 65 to 230 feet (20 to 70 m)]
6958,Red Red Wine,Q547539,who wrote the original song red red wine,fxVlSeK9xNbL,"[""Red Red Wine"" is a song originally written, ...",16,"""Red Red Wine"" is a song originally written, p...",0,[American singer Neil Diamond]


In [7]:
model_name = "Qwen/Qwen3-0.6B"

qwen_model = QALLM(model_name)

question = "What is the capital of France?"
thinking = True
answer, _ = qwen_model.generate_answer(question, thinking)
print(answer)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


The capital of France is **Paris**.
